In [ ]:
import os, json
KAGGLE_USERNAME = "suyashjaiswal005"
KAGGLE_KEY = "KGAT_c2842e6d8262e2a37714114438b4f390"
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# This adds StyleGAN/GAN-generated fake faces
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d /content/data3
!find /content/data3 -type d | head -10
!find /content/data3 -name "*.jpg" | wc -l

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:33<00:00, 119MB/s]

/content/data3
/content/data3/real_vs_fake
/content/data3/real_vs_fake/real-vs-fake
/content/data3/real_vs_fake/real-vs-fake/train
/content/data3/real_vs_fake/real-vs-fake/train/fake
/content/data3/real_vs_fake/real-vs-fake/train/real
/content/data3/real_vs_fake/real-vs-fake/test
/content/data3/real_vs_fake/real-vs-fake/test/fake
/content/data3/real_vs_fake/real-vs-fake/test/real
/content/data3/real_vs_fake/real-vs-fake/valid
140000


In [ ]:
import os, json
KAGGLE_USERNAME = "your_username"
KAGGLE_KEY = "your_key"
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d sachchitkunichetty/rvf10k
!unzip -q rvf10k.zip -d /content/data1
print("DS1 ready")

Dataset URL: https://www.kaggle.com/datasets/sachchitkunichetty/rvf10k
License(s): CC-BY-NC-SA-4.0
100% 273M/273M [00:02<00:00, 112MB/s] 

DS1 ready


In [ ]:
import shutil, random
from pathlib import Path
random.seed(42)

for split in ['train', 'val']:
    for cls in ['fake', 'real']:
        os.makedirs(f'/content/unified2/{split}/{cls}', exist_ok=True)

exts = {'.jpg','.jpeg','.png'}

def copy_dir(src, split, cls, prefix):
    files = [f for f in Path(src).rglob('*') if f.suffix.lower() in exts]
    for i, f in enumerate(files):
        shutil.copy2(str(f), f'/content/unified2/{split}/{cls}/{prefix}_{i:07d}{f.suffix}')
    return len(files)

# DS1 - rvf10k (already split)
for cls in ['fake','real']:
    n = copy_dir(f'/content/data1/rvf10k/train/{cls}', 'train', cls, 'ds1tr')
    print(f"DS1 train/{cls}: {n}")
    n = copy_dir(f'/content/data1/rvf10k/valid/{cls}', 'val', cls, 'ds1vl')
    print(f"DS1 val/{cls}: {n}")

# DS2 - Celeb-DF (manual split)
for cls in ['fake','real']:
    files = [f for f in Path(f'/content/data2/processed_data_keyframes/{cls}').rglob('*')
             if f.suffix.lower() in exts]
    random.shuffle(files)
    cut = int(len(files)*0.85)
    for i,f in enumerate(files[:cut]):
        shutil.copy2(str(f), f'/content/unified2/train/{cls}/ds2_{i:07d}{f.suffix}')
    for i,f in enumerate(files[cut:]):
        shutil.copy2(str(f), f'/content/unified2/val/{cls}/ds2v_{i:07d}{f.suffix}')
    print(f"DS2 {cls}: {cut} train / {len(files)-cut} val")

# DS3 - 140k GAN faces
for cls in ['fake','real']:
    for split,out in [('train','train'),('valid','val')]:
        n = copy_dir(f'/content/data3/real_vs_fake/real-vs-fake/{split}/{cls}', out, cls, f'ds3{split}')
        print(f"DS3 {split}/{cls}: {n}")

# Final count
print("\n=== FINAL ===")
for split in ['train','val']:
    for cls in ['fake','real']:
        print(f"{split}/{cls}: {len(os.listdir(f'/content/unified2/{split}/{cls}'))}")

DS1 train/fake: 3500
DS1 val/fake: 1500
DS1 train/real: 3500
DS1 val/real: 1500
DS2 fake: 0 train / 0 val
DS2 real: 0 train / 0 val
DS3 train/fake: 50000
DS3 valid/fake: 10000
DS3 train/real: 50000
DS3 valid/real: 10000

=== FINAL ===
train/fake: 53500
train/real: 53500
val/fake: 11500
val/real: 11500


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH = 64
IMG_SIZE = 128

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_ds = datasets.ImageFolder('/content/unified2/train', transform=train_tf)
val_ds   = datasets.ImageFolder('/content/unified2/val',   transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

print("Classes:", train_ds.class_to_idx)
print("Train:", len(train_ds), "| Val:", len(val_ds))

Classes: {'fake': 0, 'real': 1}
Train: 107000 | Val: 23000


In [ ]:
class DeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),    nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2),                   nn.Dropout(0.1),
            nn.Conv2d(32, 64, 3, padding=1),   nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),   nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),                   nn.Dropout(0.1),
            nn.Conv2d(64, 128, 3, padding=1),  nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                   nn.Dropout(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = DeepfakeCNN().to(DEVICE)

EPOCHS = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        t_correct += (out.argmax(1) == labels).sum().item()

    model.eval()
    v_loss, v_correct = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            v_loss += criterion(out, labels).item()
            v_correct += (out.argmax(1) == labels).sum().item()

    t_acc = t_correct / len(train_ds)
    v_acc = v_correct / len(val_ds)
    history['train_loss'].append(t_loss/len(train_loader))
    history['val_loss'].append(v_loss/len(val_loader))
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    scheduler.step(v_acc)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {t_loss/len(train_loader):.4f} Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss/len(val_loader):.4f} Acc: {v_acc:.4f}")

print(f"\nBest Val Acc: {max(history['val_acc']):.4f}")
torch.save(model.state_dict(), 'deepfake_cnn_v3.pth')
print("Saved: deepfake_cnn_v3.pth")

Epoch 01/10 | Train Loss: 0.6707 Acc: 0.6059 | Val Loss: 0.5650 Acc: 0.7099
Epoch 02/10 | Train Loss: 0.5092 Acc: 0.7536 | Val Loss: 0.4027 Acc: 0.8182
Epoch 03/10 | Train Loss: 0.3889 Acc: 0.8280 | Val Loss: 0.3261 Acc: 0.8576
Epoch 04/10 | Train Loss: 0.3065 Acc: 0.8708 | Val Loss: 0.2132 Acc: 0.9135
Epoch 05/10 | Train Loss: 0.2497 Acc: 0.8984 | Val Loss: 0.1882 Acc: 0.9271
Epoch 06/10 | Train Loss: 0.2092 Acc: 0.9161 | Val Loss: 0.1337 Acc: 0.9461
Epoch 07/10 | Train Loss: 0.1832 Acc: 0.9286 | Val Loss: 0.1573 Acc: 0.9328
Epoch 08/10 | Train Loss: 0.1629 Acc: 0.9378 | Val Loss: 0.0919 Acc: 0.9654
Epoch 09/10 | Train Loss: 0.1464 Acc: 0.9438 | Val Loss: 0.1459 Acc: 0.9420
Epoch 10/10 | Train Loss: 0.1338 Acc: 0.9493 | Val Loss: 0.0800 Acc: 0.9708

Best Val Acc: 0.9708
Saved: deepfake_cnn_v3.pth


In [ ]:
!pip install -q gradio
import gradio as gr

CLASS_NAMES = train_ds.class_to_idx  # check: fake=0, real=1
model.eval()

def detect_deepfake(image):
    tensor = val_tf(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    fake_conf = probs[0].item()
    real_conf = probs[1].item()
    label = "🔴 FAKE" if fake_conf > real_conf else "🟢 REAL"
    return {label: max(fake_conf, real_conf),
            "Fake probability": fake_conf,
            "Real probability": real_conf}

gr.Interface(
    fn=detect_deepfake,
    inputs=gr.Image(type="pil", label="Upload Face Image"),
    outputs=gr.Label(label="Result"),
    title="Deepfake Detector v3",
    description="Trained on: GAN faces + face swaps + real faces. 96.9% accuracy."
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://698f8bf959e77fb810.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q gradio opencv-python-headless

import cv2
import numpy as np
import gradio as gr
from PIL import Image

# OpenCV face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def crop_face(pil_img):
    img = np.array(pil_img.convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(60,60))
    if len(faces) == 0:
        return pil_img  # no face found, use full image
    x, y, w, h = max(faces, key=lambda f: f[2]*f[3])  # largest face
    pad = int(0.2 * max(w, h))
    x1 = max(0, x-pad); y1 = max(0, y-pad)
    x2 = min(img.shape[1], x+w+pad); y2 = min(img.shape[0], y+h+pad)
    cropped = img[y1:y2, x1:x2]
    return Image.fromarray(cropped)

model.eval()

def detect_deepfake(image):
    face = crop_face(image)
    tensor = val_tf(face).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    fake_conf = probs[0].item()
    real_conf = probs[1].item()
    label = "🔴 FAKE" if fake_conf > real_conf else "🟢 REAL"
    return {label: max(fake_conf, real_conf),
            "Fake probability": fake_conf,
            "Real probability": real_conf}

gr.Interface(
    fn=detect_deepfake,
    inputs=gr.Image(type="pil", label="Upload Face Image"),
    outputs=gr.Label(label="Result"),
    title="Deepfake Detector v3 + Face Crop",
    description="Auto-crops face before detection."
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d046ceaf4c5eac19c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
